In [1]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

### Simulation

In [2]:
ds = 'Simulation'

data_dir = '../../../data/processed/Simulations'
fs = list(sorted(os.listdir(data_dir)))
fs

['rna-0.1-12-adt-2.1',
 'rna-0.2-10-adt-2.2',
 'rna-0.3-7-adt-2.3',
 'rna-0.4-5-adt-2.4',
 'rna-0.5-4-adt-2.5']

In [3]:
for fn in fs:
    for seed in [1,2,3]:
        _dir = join(data_dir, fn, str(seed))
        ad_mult_rna = sc.read_h5ad(join(_dir, 'ad_rna1.h5ad'))
        ad_mult_adt = sc.read_h5ad(join(_dir, 'ad_adt1.h5ad'))
        ad_test_rna = sc.read_h5ad(join(_dir, 'ad_rna2.h5ad'))
        ad_test_adt = sc.read_h5ad(join(_dir, 'ad_adt2.h5ad'))
        
        ad_mult_rna.X = ad_mult_rna.layers['counts'].copy()
        ad_mult_adt.X = ad_mult_adt.layers['counts'].copy()
        ad_test_rna.X = ad_test_rna.layers['counts'].copy()
        ad_test_adt.X = ad_test_adt.layers['counts'].copy()

        # 添加 src 标签
        ad_mult_rna.obs['src'] = 's1'
        ad_mult_adt.obs['src'] = 's1'
        ad_test_rna.obs['src'] = 's2'
        ad_test_adt.obs['src'] = 's3'

        # 给 test 的 obs_names 加前缀
        ad_test_rna.obs_names = [f"test-mod1-{i}" for i in range(ad_test_rna.n_obs)]
        ad_test_adt.obs_names = [f"test-mod2-{i}" for i in range(ad_test_adt.n_obs)]

        # concat
        ad_rna = sc.concat([ad_mult_rna, ad_test_rna])
        ad_atac = sc.concat([ad_mult_adt, ad_test_adt])

        # scale + pca + harmony
        sc.pp.scale(ad_rna)
        sc.pp.pca(ad_rna, n_comps=50)
        ad_rna.obsm['X_pca_har'] = harmony(
            ad_rna.obsm['X_pca'],
            ad_rna.obs['src'].to_list(),
            use_gpu=True
        )
        utls.split_adata_ob([ad_mult_rna, ad_test_rna], ad_rna, ob='obsm', key='X_pca_har')

        sc.pp.scale(ad_atac)
        sc.pp.pca(ad_atac, n_comps=50)
        ad_atac.obsm['X_pca_har'] = harmony(
            ad_atac.obsm['X_pca'],
            ad_atac.obs['src'].to_list(),
            use_gpu=True
        )
        utls.split_adata_ob([ad_mult_adt, ad_test_adt], ad_atac, ob='obsm', key='X_pca_har')
        
        input_dict = { 
            'rna':  [ad_mult_rna,  ad_test_rna,  None],
            'adt':  [ad_mult_adt, None       ,  ad_test_adt],
        }
        input_key = 'X_pca_har'
        
        cm_ratio = 0.5 
        model = SpaMosaic(
            modBatch_dict=input_dict, input_key=input_key,
            batch_key='src', 
            intra_knns=10, inter_knn_base=10, 
            w_g=0.8,
            seed=1234, 
            device='cuda:0'
        )

        model.train(net='wlgcn', lr=0.01, T=0.1, n_epochs=100)
        
        ad_embs = model.infer_emb(input_dict, emb_key='emb', final_latent_key='merged_emb')
        ad_mosaic = sc.concat(ad_embs)
        ad_mosaic = utls.get_umap(ad_mosaic, use_reps=['merged_emb'])
        
        os.makedirs(f'../../../results/embeddings/SpaMosaic/Simulations/{fn}/{seed}', exist_ok=True)
        df_emb.to_csv(f'../../../results/embeddings/SpaMosaic/Simulations/{fn}/{seed}/df_emb.csv')

Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
Reach convergence after 8 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
Reach convergence after 9 iteration(s).
batch0: ['rna', 'adt']
batch1: ['rna']
batch2: ['adt']
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on averag

100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 49.86it/s]
/home/yanxh/anaconda3/envs/spamosaic-env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
Reach convergence after 7 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
Reach convergence after 8 iteration(s).
batch0: ['rna', 'adt']
batch1: ['rna']
batch2: ['adt']
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 e

100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [00:01<00:00, 53.79it/s]


Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
Reach convergence after 7 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
Reach convergence after 7 iteration(s).
batch0: ['rna', 'adt']
batch1: ['rna']
batch2: ['adt']
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighb

...

100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 49.13it/s]


Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
Reach convergence after 4 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
	Completed 10 / 10 iteration(s).
batch0: ['rna', 'adt']
batch1: ['rna']
batch2: ['adt']
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculat

100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 49.09it/s]


Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
Reach convergence after 6 iteration(s).
Use GPU mode.
	Initialization is completed.
	Completed 1 / 10 iteration(s).
	Completed 2 / 10 iteration(s).
	Completed 3 / 10 iteration(s).
	Completed 4 / 10 iteration(s).
	Completed 5 / 10 iteration(s).
	Completed 6 / 10 iteration(s).
	Completed 7 / 10 iteration(s).
	Completed 8 / 10 iteration(s).
	Completed 9 / 10 iteration(s).
Reach convergence after 9 iteration(s).
batch0: ['rna', 'adt']
batch1: ['rna']
batch2: ['adt']
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 edges, 1296 cells.
10.0000 neighbors per cell on average.
------Calculating spatial graph...
The graph contains 12960 e

100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 48.38it/s]
